# Notebook 03 — Evaluation Analysis

Deep-dive into the evaluation results across all experiments:
- Metric distributions and correlations
- Experiment leaderboard and delta analysis
- Per-example score breakdown
- Cost and latency profiling

**Prerequisites:** Run `python scripts/run_baseline.py` and `python scripts/run_experiments.py` first.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().absolute().parent / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
%matplotlib inline

from llm_evals_lab.config import load_config
from llm_evals_lab.observability.run_store import RunStore
from llm_evals_lab.experiments.compare import ExperimentComparison

cfg = load_config()
run_store = RunStore(cfg.runs_dir(), cfg.tables_dir())
df = run_store.to_dataframe()

if df.empty:
    print('No run data found. Run the pipeline scripts first.')
else:
    print(f'Loaded {len(df)} run records')
    print(f'Experiments: {sorted(df.experiment_id.unique().tolist())}')

## 1. Metric Summary Statistics

In [ ]:
if not df.empty:
    metric_cols = ['overall_score', 'answer_relevance', 'groundedness',
                   'citation_coverage', 'faithfulness_proxy', 'hit_at_k',
                   'context_recall', 'hallucination_risk']
    available = [c for c in metric_cols if c in df.columns]
    print('Metric summary (all experiments):')
    display(df[available].describe().T.round(4))

## 2. Experiment Leaderboard

In [ ]:
if not df.empty:
    comp = ExperimentComparison(df)
    lb = comp.leaderboard()
    print('Leaderboard:')
    display(lb)

In [ ]:
if not df.empty and 'experiment_id' in df.columns:
    metric_cols = ['overall_score', 'groundedness', 'citation_coverage', 'hit_at_k']
    available = [c for c in metric_cols if c in df.columns]
    agg = df.groupby('experiment_id')[available].mean()

    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 4))
    if len(available) == 1:
        axes = [axes]
    for ax, metric in zip(axes, available):
        sorted_agg = agg[metric].sort_values(ascending=True)
        ax.barh(sorted_agg.index, sorted_agg.values, color='#4C72B0')
        ax.set_xlim(0, 1)
        ax.set_title(metric)
        ax.set_xlabel('Mean Score')
        for i, (idx, val) in enumerate(sorted_agg.items()):
            ax.text(val + 0.01, i, f'{val:.3f}', va='center', fontsize=9)

    plt.suptitle('Experiment Comparison by Metric', y=1.02)
    plt.tight_layout()
    plt.savefig('../results/figures/03_experiment_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Per-Example Score Heatmap

In [ ]:
if not df.empty and 'example_id' in df.columns:
    pivot = df.pivot_table(
        index='example_id',
        columns='experiment_id',
        values='overall_score',
        aggfunc='mean'
    ).fillna(0)

    fig, ax = plt.subplots(figsize=(max(8, len(pivot.columns) * 1.5), max(5, len(pivot) * 0.5)))
    cax = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)
    plt.colorbar(cax, ax=ax, label='Overall Score')
    ax.set_title('Overall Score per Example × Experiment')
    plt.tight_layout()
    plt.savefig('../results/figures/03_score_heatmap.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Metric Correlation Analysis

In [ ]:
if not df.empty:
    corr_cols = ['overall_score', 'answer_relevance', 'groundedness',
                 'citation_coverage', 'hit_at_k', 'context_recall']
    available = [c for c in corr_cols if c in df.columns]
    corr = df[available].corr().round(2)

    fig, ax = plt.subplots(figsize=(8, 6))
    cax = ax.imshow(corr.values, cmap='coolwarm', vmin=-1, vmax=1)
    ax.set_xticks(range(len(available)))
    ax.set_xticklabels(available, rotation=45, ha='right')
    ax.set_yticks(range(len(available)))
    ax.set_yticklabels(available)
    for i in range(len(available)):
        for j in range(len(available)):
            ax.text(j, i, f'{corr.values[i,j]:.2f}', ha='center', va='center', fontsize=9)
    plt.colorbar(cax)
    ax.set_title('Metric Correlation Matrix')
    plt.tight_layout()
    plt.show()

## 5. Latency and Cost Profile

In [ ]:
if not df.empty and 'latency_ms' in df.columns:
    print('Latency statistics by experiment:')
    lat_stats = df.groupby('experiment_id')['latency_ms'].agg(['mean', 'median', 'std', 'max']).round(2)
    display(lat_stats)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    experiments = df['experiment_id'].unique()
    for exp in experiments:
        exp_df = df[df['experiment_id'] == exp]
        axes[0].hist(exp_df['latency_ms'].dropna(), alpha=0.6, label=exp, bins=15)
    axes[0].set_title('Latency Distribution by Experiment')
    axes[0].set_xlabel('Latency (ms)')
    axes[0].legend()

    if 'overall_score' in df.columns:
        for exp in experiments:
            exp_df = df[df['experiment_id'] == exp].dropna(subset=['latency_ms', 'overall_score'])
            axes[1].scatter(exp_df['latency_ms'], exp_df['overall_score'], alpha=0.6, label=exp, s=30)
        axes[1].set_title('Latency vs Quality')
        axes[1].set_xlabel('Latency (ms)')
        axes[1].set_ylabel('Overall Score')
        axes[1].legend()

    plt.tight_layout()
    plt.show()